In [1]:
import sys
import importlib
sys.path.append('../')  # Adjust the path as needed

import utilities.functions as functions

# Reload the module to reflect the changes
importlib.reload(functions)

<module 'utilities.functions' from '/Users/xuechenkan/potts_model_test/ms1/../utilities/functions.py'>

In [2]:
IN_syn_pairs = ['G140S-Q148H', 'Y143C-S230R']
PR_syn_pairs = ['D30N-N88D', 'V32I-I47V']
RT_syn_pairs = ['K101E-G190S', 'K103N-P225H']

IN_seq_path = 'IN/data/in.reduce4.seq'
PR_seq_path = 'PR/data/pr.exper.reduce4.seq'    
RT_seq_path = 'RT/data/rt.reduce4.seq'

IN_all_seq = functions.read_seq('IN/data/in.reduce4.seq')
PR_all_seq = functions.read_seq('PR/data/pr.exper.reduce4.seq')
RT_all_seq = functions.read_seq('RT/data/rt.reduce4.seq')

IN_consensus = 'IN/data/in.consensus.reduce4.seq'
PR_consensus = 'PR/data/pr.consensus.reduce4.seq'
RT_consensus = 'RT/data/rt.consensus.reduce4.seq'

IN_redux = functions.get_redu_dict('IN/data/in.reduce4.redux',1)
PR_redux = functions.get_redu_dict('PR/data/pr.reduce4.redux',0)
RT_redux = functions.get_redu_dict('RT/data/rt.reduce4.redux',0)

IN_J = functions.load_J_dict('IN/data/J.npy',1,263)
PR_J = functions.load_J_dict('PR/data/J_PR.npy',1,99)
RT_J = functions.load_J_dict('RT/data/J_RT.npy',39,226)

IN_all_seq_unreduced = functions.read_seq('IN/data/in.fullseq')
PR_all_seq_unreduced = functions.read_seq('PR/data/pr.exper.fullseq')


# Convert synergistic pairs to reduced format
IN_syn_pairs_reduced = functions.pairs_to_reduced(IN_redux, IN_syn_pairs)
PR_syn_pairs_reduced = functions.pairs_to_reduced(PR_redux, PR_syn_pairs)
RT_syn_pairs_reduced = functions.pairs_to_reduced(RT_redux, RT_syn_pairs)

print("IN reduced pairs:", IN_syn_pairs_reduced)
print("PR reduced pairs:", PR_syn_pairs_reduced)
print("RT reduced pairs:", RT_syn_pairs_reduced)

IN reduced pairs: ['C140D-D148B', 'D143A-D230C']
PR reduced pairs: ['B30D-B88C', 'C32D-A47B']
RT reduced pairs: ['C101A-D190C', 'C103B-D225A']


In [3]:
IN_seq_mut_df = functions.analyze_sequences_mutations(IN_consensus, IN_seq_path)
IN_seq_mut_df.head()

,Sequence,Mutations,Mutations_count
0,ABAAABABACBBAACBDBABBDBDBBBAAAAAABAAABDAABAABA...,"[C7A, A11B, C31A, C50B, A72D, A101B, C124A, B1...",13
1,ABAAABCBACBBAACBDBABCDBDABBAAACAABAAABDAABAABA...,"[A11B, B21C, B25A, D119A, C122B, D125A, D148C,...",13
2,ABAAABCBACBBAACBDBABCDBDABBAAACAABAAABDAABAABA...,"[A11B, B21C, B25A, D119A, C122B, D125A, C140D,...",13
3,ABAAABCBACABAACBABABBDBDBBBBAACAABAAABAAABAABA...,"[D17A, A28B, D39A, D119C, C122B, C124A, D125A,...",9
4,ABAAABCBABABAACBDBABBDBDBBBAAACAABAAABDAABAABA...,"[C10B, A101B, B106A, A155C, C156A, C165A, C195...",9


In [ ]:
import pandas as pd

# Load J matrix and create J_dict for IN
IN_J_file = 'IN/data/in.reduce4.J'
IN_max_position = len(IN_all_seq[0])

# Get consensus sequence
with open(IN_consensus, 'r') as f:
    IN_consensus_seq = f.read().strip()

# Calculate energies for each reduced pair
IN_results = []

for pair in IN_syn_pairs_reduced:
    mut1, mut2 = pair.split('-')
    de1_consensus = functions.calculate_delta_e(mut1, IN_consensus_seq, IN_J, 1, 263)
    de2_consensus = functions.calculate_delta_e(mut2, IN_consensus_seq, IN_J, 1, 263)
    de12_consensus = functions.calculate_delta_e_double(mut1, mut2, IN_consensus_seq, IN_J, 1, 263)

    for idx, row in IN_seq_mut_df.iterrows():
        seq = row['Sequence']
        mutations = row['Mutations']
        # mutations_unreduced = functions.reduced_to_unreduced_list(IN_redux, mutations, IN_all_seq_unreduced)
        de1 = functions.calculate_delta_e(mut1, seq, IN_J, 1, 263)
        de2 = functions.calculate_delta_e(mut2, seq, IN_J, 1, 263)
        de12 = functions.calculate_delta_e_double(mut1, mut2, seq, IN_J, 1, 263)
        
        if de12 is None:
            # print(f"Warning: Could not calculate de12 for row {idx}. Skipping.")
            continue
        dde = de12 - de1 - de2
        
        # FLIPPPPPPPPPPP
        if (de1_consensus-de2_consensus) * (de1-de2) < 0 :
            IN_results.append({
                'Epistatis_type': 'FLIP',
                'original_pair': IN_syn_pairs[IN_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde

            })
        
        elif (de12 > de1 or de12 > de2) and not (de12 > de1 and de12 > de2):
            IN_results.append({
                'Epistatis_type': 'COMPENSATORY',
                'original_pair': IN_syn_pairs[IN_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde
            })
        
        elif de12 > de1 and de12 > de2:
            IN_results.append({
                'Epistatis_type': 'RESCUE',
                'original_pair': IN_syn_pairs[IN_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde
            })
        else:
            IN_results.append({
                'Epistatis_type': 'OTHER',
                'original_pair': IN_syn_pairs[IN_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde
            })
        
        # Group results by original pair and epistatic type, keeping only first 2 of each type
        pair_type_counts = {}
        filtered_indices = []


# Display results
IN_energy_df = pd.DataFrame(IN_results)
print(IN_energy_df)

     Epistatis_type original_pair  \
0              FLIP   G140S-Q148H   
1      COMPENSATORY   G140S-Q148H   
2      COMPENSATORY   G140S-Q148H   
3            RESCUE   G140S-Q148H   
4            RESCUE   G140S-Q148H   
...             ...           ...   
1904           FLIP   Y143C-S230R   
1905          OTHER   Y143C-S230R   
1906   COMPENSATORY   Y143C-S230R   
1907           FLIP   Y143C-S230R   
1908           FLIP   Y143C-S230R   

                                              mutations       de1       de2  \
0     [E10D, L101I, G106A, N155H, K156N, V165I, S195... -7.654879 -7.701222   
1     [V32I, S39C, M50I, T112I, I113V, T124N, N155H,... -8.819580 -7.610831   
2     [E11D, V31M, V32I, S39C, I72V, E92Q, L101I, K1... -9.428816 -7.822751   
3         [R20K, A23V, I72V, E92Q, T124A, I204V, T206S] -7.312345 -5.870149   
4                [E11D, A21T, A23V, D25E, E157Q, V201I] -5.459156 -4.374571   
...                                                 ...       ...       ...   
19

In [ ]:
# import csv

# IN_reduced_lists = [
#     ['C10B', 'A101B', 'B106A', 'A155C', 'C156A', 'C165A', 'C195B', 'B201A', 'C220D'],
#     ['A11B', 'C31A', 'A101B', 'B111C', 'D119C', 'C232A', 'B256A'],
#     ['A32B', 'D39A', 'C50A', 'D112B', 'C113D', 'C124D', 'A155C', 'A163D', 'B201A', 'C234A', 'D255A'],
#     ['A11B', 'C31B', 'A32B', 'D39A', 'A72D', 'A92C', 'A101B', 'B111D', 'D119C', 'B135C', 'A155C', 'A193C', 'B201A', 'D218C'],
#     ['B20A', 'B23D', 'A72D', 'A92C', 'C124A', 'B204A', 'B206A'],
#     ['A11B', 'B21C', 'B23D', 'B25A', 'B157A', 'B201A'],
#     ['C7A', 'A11B', 'C31A', 'C50B', 'A72D', 'A101B', 'C124A', 'B135C', 'C140D', 'D148B', 'B200A', 'B201A', 'C220B'],
#     ['D17A', 'A28B', 'D39A', 'D119C', 'C122B', 'C124A', 'D125A', 'C140D', 'D148B'],
#     ['A72D', 'C124A', 'C156A', 'A167B'],
#     ['A14B', 'D22C', 'A32B', 'A37B', 'D39A', 'A101B', 'A163D', 'B201A', 'D253A'],
#     ['D17A', 'B20A', 'A28B', 'D39A', 'C124A', 'D125A', 'A155C', 'B201A', 'C208A'],
#     ['C31A', 'A54B', 'C113D', 'C124A', 'D125A', 'B201A', 'C215B']
# ]

# IN_unreduced_lists = []
# for li in IN_reduced_lists:
#     IN_unreduced_lists.append(functions.reduced_to_unreduced_list(IN_redux, li, IN_all_seq_unreduced))
# print(IN_unreduced_lists)
    
# # Write the list to a CSV file
# with open('IN_unreduced_lists.csv', 'w', newline='') as csvfile:
#     writer = csv.writer(csvfile)
#     writer.writerow(IN_unreduced_lists)

[['E10D', 'L101I', 'G106A', 'N155H', 'K156N', 'V165I', 'S195C', 'V201I', 'I220V'], ['E11D', 'V31I', 'L101I', 'K111T', 'S119P', 'D232E', 'D256E'], ['V32I', 'S39C', 'M50I', 'T112I', 'I113V', 'T124N', 'N155H', 'G163R', 'V201I', 'L234V', 'S255K'], ['E11D', 'V31M', 'V32I', 'S39C', 'I72V', 'E92Q', 'L101I', 'K111R', 'S119P', 'I135V', 'N155H', 'G193E', 'V201I', 'T218S'], ['R20K', 'A23V', 'I72V', 'E92Q', 'T124A', 'I204V', 'T206S'], ['E11D', 'A21T', 'A23V', 'D25E', 'E157Q', 'V201I'], ['K7Q', 'E11D', 'V31I', 'M50L', 'I72V', 'L101I', 'T124A', 'I135V', 'G140S', 'Q148H', 'I200L', 'V201I', 'I220L'], ['S17N', 'L28I', 'S39C', 'S119P', 'T122I', 'T124A', 'T125A', 'G140S', 'Q148H'], ['I72V', 'T124A', 'K156N', 'D167E'], ['K14R', 'M22L', 'V32I', 'V37I', 'S39C', 'L101I', 'G163R', 'V201I', 'D253E'], ['S17N', 'R20K', 'L28I', 'S39C', 'T124A', 'T125A', 'N155H', 'V201I', 'I208L'], ['V31I', 'V54I', 'I113V', 'T124A', 'T125A', 'V201I', 'K215N']]


In [5]:
PR_seq_mut_df = functions.analyze_sequences_mutations(PR_consensus, PR_seq_path)

# Get consensus sequence
with open(PR_consensus, 'r') as f:
    PR_consensus_seq = f.read().strip()

# Calculate energies for each reduced pair
PR_results = []

for pair in PR_syn_pairs_reduced:
    mut1, mut2 = pair.split('-')
    de1_consensus = functions.calculate_delta_e(mut1, PR_consensus_seq, PR_J, 1, 99)
    de2_consensus = functions.calculate_delta_e(mut2, PR_consensus_seq, PR_J, 1, 99)
    de12_consensus = functions.calculate_delta_e_double(mut1, mut2, PR_consensus_seq, PR_J, 1, 99)

    for idx, row in PR_seq_mut_df.iterrows():
        seq = row['Sequence']
        mutations = row['Mutations']
        de1 = functions.calculate_delta_e(mut1, seq, PR_J, 1, 99)
        de2 = functions.calculate_delta_e(mut2, seq, PR_J, 1, 99)
        de12 = functions.calculate_delta_e_double(mut1, mut2, seq, PR_J, 1, 99)
        
        if de12 is None:
            continue
        dde = de12 - de1 - de2
        
        # FLIP
        if (de1_consensus-de2_consensus) * (de1-de2) < 0:
            PR_results.append({
                'Epistatis_type': 'FLIP',
                'original_pair': PR_syn_pairs[PR_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde
            })
        
        elif (de12 > de1 or de12 > de2) and not (de12 > de1 and de12 > de2):
            PR_results.append({
                'Epistatis_type': 'COMPENSATORY',
                'original_pair': PR_syn_pairs[PR_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde
            })
        
        elif de12 > de1 and de12 > de2:
            PR_results.append({
                'Epistatis_type': 'RESCUE',
                'original_pair': PR_syn_pairs[PR_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde
            })
        else:
            PR_results.append({
                'Epistatis_type': 'OTHER',
                'original_pair': PR_syn_pairs[PR_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde
            })

# Display results
PR_energy_df = pd.DataFrame(PR_results)
print(PR_energy_df)

      Epistatis_type original_pair  \
0              OTHER     D30N-N88D   
1               FLIP     D30N-N88D   
2               FLIP     D30N-N88D   
3               FLIP     D30N-N88D   
4       COMPENSATORY     D30N-N88D   
...              ...           ...   
10520           FLIP     V32I-I47V   
10521          OTHER     V32I-I47V   
10522          OTHER     V32I-I47V   
10523          OTHER     V32I-I47V   
10524   COMPENSATORY     V32I-I47V   

                                               mutations       de1       de2  \
0      [D10C, C14D, D35C, D36C, C37B, C54D, A63D, B64... -5.241160 -5.643820   
1      [C32D, D35C, C37B, C46D, A47B, A63D, C73B, D77... -7.670992 -6.936328   
2      [D15C, C32D, C37B, C46D, A47B, A63D, C82B, A92... -6.982893 -6.548922   
3             [D10C, D15C, A24D, C54D, A63D, A71B, C82B] -6.005604 -5.761354   
4                                     [D15C, A63D, B64D] -2.745276 -4.643936   
...                                                  ...       

In [25]:
import csv

PR_reduced_lists = [
    ['C32D', 'D35C', 'C37B', 'C46D', 'A47B', 'A63D', 'C73B', 'D77C', 'C90B', 'A93B'],
    ['D15C', 'C32D', 'C37B', 'C46D', 'A47B', 'A63D', 'C82B', 'A92C', 'A93B'],
    ['D15C', 'A63D', 'B64D'],
    ['D36A', 'A63D'],
    ['D13B', 'D15C', 'D35C', 'A63D', 'A74C'],
    ['C62D', 'A63D', 'C67B', 'C69A', 'D77C', 'A93B'],
    ['D10C', 'C14D', 'D35C', 'D36C', 'C37B', 'C54D', 'A63D', 'B64C', 'A71C', 'C82A', 'C90B', 'A93B'],
    ['B41C', 'C54D', 'C62D', 'A63D', 'D77C', 'C82B'],
    ['D10C', 'C37A', 'B41C', 'C46D', 'C54D', 'C62D', 'A63D', 'A71B', 'D77C', 'C82B', 'C90B', 'A93B'],
    ['D10A', 'A24D', 'D35C', 'C37B', 'C46D', 'A63D', 'C82B', 'A92D'],
    ['D13B', 'C14D', 'A33C', 'D36A', 'C46D', 'C62D', 'A63D', 'A71C', 'C73B', 'C90B', 'A93B'],
    ['D10A', 'D13B', 'C19B', 'C46D', 'A63D', 'C73A', 'C90B']
]

PR_unreduced_lists = []
for li in PR_reduced_lists:
    PR_unreduced_lists.append(functions.reduced_to_unreduced_list(PR_redux, li, PR_all_seq_unreduced))
print(PR_unreduced_lists)
    
# Write the list to a CSV file
with open('PR_unreduced_lists.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(PR_unreduced_lists)

[['V32I', 'E35D', 'N37D', 'M46I', 'I47V', 'L63P', 'G73S', 'V77I', 'L90M', 'I93L'], ['I15V', 'V32I', 'N37D', 'M46I', 'I47V', 'L63P', 'V82A', 'Q92R', 'I93L'], ['I15V', 'L63P', 'I64L'], ['M36I', 'L63P'], ['I13V', 'I15V', 'E35D', 'L63P', 'T74S'], ['I62V', 'L63P', 'C67F', 'H69K', 'V77I', 'I93L'], ['L10I', 'K14R', 'E35D', 'M36V', 'N37D', 'I54V', 'L63P', 'I64V', 'A71T', 'V82T', 'L90M', 'I93L'], ['R41K', 'I54V', 'I62V', 'L63P', 'V77I', 'V82A'], ['L10I', 'N37S', 'R41K', 'M46I', 'I54V', 'I62V', 'L63P', 'A71V', 'V77I', 'V82A', 'L90M', 'I93L'], ['L10V', 'L24I', 'E35D', 'N37D', 'M46I', 'L63P', 'V82A', 'Q92K'], ['I13V', 'K14R', 'L33I', 'M36I', 'M46I', 'I62V', 'L63P', 'A71T', 'G73S', 'L90M', 'I93L'], ['L10V', 'I13V', 'L19I', 'M46I', 'L63P', 'G73C', 'L90M']]


In [6]:
RT_seq_mut_df = functions.analyze_sequences_mutations(RT_consensus, RT_seq_path)

# Get consensus sequence
with open(RT_consensus, 'r') as f:
    RT_consensus_seq = f.read().strip()

# Calculate energies for each reduced pair
RT_results = []

for pair in RT_syn_pairs_reduced:
    mut1, mut2 = pair.split('-')
    de1_consensus = functions.calculate_delta_e(mut1, RT_consensus_seq, RT_J, 39, 226)
    de2_consensus = functions.calculate_delta_e(mut2, RT_consensus_seq, RT_J, 39, 226)
    de12_consensus = functions.calculate_delta_e_double(mut1, mut2, RT_consensus_seq, RT_J, 39, 226)

    for idx, row in RT_seq_mut_df.iterrows():
        seq = row['Sequence']
        mutations = row['Mutations']
        de1 = functions.calculate_delta_e(mut1, seq, RT_J, 39, 226)
        de2 = functions.calculate_delta_e(mut2, seq, RT_J, 39, 226)
        de12 = functions.calculate_delta_e_double(mut1, mut2, seq, RT_J, 39, 226)
        
        if de12 is None:
            continue
        dde = de12 - de1 - de2
        
        # FLIP
        if (de1_consensus-de2_consensus) * (de1-de2) < 0:
            RT_results.append({
                'Epistatis_type': 'FLIP',
                'original_pair': RT_syn_pairs[RT_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde
            })
        
        elif (de12 > de1 or de12 > de2) and not (de12 > de1 and de12 > de2):
            RT_results.append({
                'Epistatis_type': 'COMPENSATORY',
                'original_pair': RT_syn_pairs[RT_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde
            })
        
        elif de12 > de1 and de12 > de2:
            RT_results.append({
                'Epistatis_type': 'RESCUE',
                'original_pair': RT_syn_pairs[RT_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde
            })
        else:
            RT_results.append({
                'Epistatis_type': 'OTHER',
                'original_pair': RT_syn_pairs[RT_syn_pairs_reduced.index(pair)],
                'mutations': mutations,
                'de1': de1,
                'de2': de2,
                'de12': de12,
                'dde': dde
            })

# Display results
RT_energy_df = pd.DataFrame(RT_results)
print(RT_energy_df)

      Epistatis_type original_pair  \
0              OTHER   K101E-G190S   
1               FLIP   K101E-G190S   
2               FLIP   K101E-G190S   
3               FLIP   K101E-G190S   
4               FLIP   K101E-G190S   
...              ...           ...   
25397   COMPENSATORY   K103N-P225H   
25398   COMPENSATORY   K103N-P225H   
25399   COMPENSATORY   K103N-P225H   
25400   COMPENSATORY   K103N-P225H   
25401   COMPENSATORY   K103N-P225H   

                                               mutations       de1       de2  \
0             [D10C, D45B, C65B, C66A, A84C, B85C, A97C] -5.286482 -5.422381   
1                [D1A, C65B, C135B, B136D, D169B, D173C] -5.306311 -5.231634   
2            [D1B, D10C, D36B, D45B, B85C, B139A, D162A] -3.931431 -3.129288   
3      [D11A, D22B, A29D, C32D, C65B, B85C, A97C, B12... -4.946983 -4.570993   
4                       [D45B, D52C, C65B, D173C, D177C] -5.044549 -4.964806   
...                                                  ...       